# Magic Eraser: run B0 in Colab

Runtime → Change runtime type → **T4 GPU** before starting.

This notebook only calls scripts in `src/`. Put new logic in `src/`, not here.

In [ ]:
!nvidia-smi

## 1. Get the code
Replace the URL with your repo. For a private repo, use a GitHub token or upload a zip.

In [ ]:
!git clone https://github.com/Norrozo/5810finalproject.git
%cd 5810finalproject

## 2. Install
SAM 2 takes a few minutes to install.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "git+https://github.com/facebookresearch/sam2.git"

## 3. (Optional) Save outputs to Google Drive
Colab wipes files when the session ends. Linking `runs/` and `results/` to Drive keeps them, and lets interrupted runs resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/magic-eraser/runs
!rm -rf runs && ln -s /content/drive/MyDrive/magic-eraser/runs runs

## 4. Download DAVIS 2017 (480p, about 800 MB)
If the link has moved, get "TrainVal - Images and Annotations (480p)" from davischallenge.org.

In [ ]:
!wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip
!unzip -q DAVIS-2017-trainval-480p.zip && rm DAVIS-2017-trainval-480p.zip
!ls DAVIS

## 5. Build the synthetic benchmark
Look at a few composites before running B0. If a pair looks broken, change it in `configs/eval_clips.txt` now. After B0, the list stays frozen.

In [ ]:
!python -m src.make_synthetic --config configs/B0.yaml --davis-root DAVIS

In [ ]:
from IPython.display import Image, display
import glob
for clip in sorted(glob.glob('data/*'))[:3]:
    display(Image(filename=f'{clip}/frames/00000.png', width=400))

## 6. Run B0

In [ ]:
!python -m src.run_pipeline --config configs/B0.yaml

## 7. Results

In [ ]:
!python -m src.summarize

In [ ]:
display(Image(filename='runs/B0/dog_on_soapbox/stills.png'))

## 8. Try your own video
Upload a short clip, find the object's pixel coordinates in the first frame, then run:

In [ ]:
# !python -m src.run_pipeline --config configs/B0.yaml --video my_clip.mp4 --name my_clip --click 320,240

## 9. Save results
`results/results.csv` is small. Download it or commit it to the repo after each run.

In [ ]:
from google.colab import files
files.download('results/results.csv')